# Segmentación de distritos de California y regresión del valor de vivienda

**Proyecto académico de Estadística 4**

Análisis exploratorio, construcción de indicadores por hogar, winsorización, estandarización, clustering K-means y regresión multivariante para caracterizar distritos de California.

> Los datos son históricos (1990). El análisis es educativo y no constituye asesoría de inversión.

##0. Carga y selección de la muestra

Semilla utilizada: 5759

In [ ]:

# ============================================================
# PRÁCTICA COMPUTACIONAL 3
# Clusterización y Regresión Multivariante
# Estadística 4
# ============================================================

import pandas as pd
import numpy as np

# Semilla fija para reproducibilidad del análisis
SEMILLA = 5759

# 1. Cargar la base de datos original de Google Colab
ruta = "/content/sample_data/california_housing_train.csv"
df_original = pd.read_csv(ruta)

# 2. Revisar la base original
print("BASE DE DATOS ORIGINAL")
print("Número de filas:", df_original.shape[0])
print("Número de columnas:", df_original.shape[1])

display(df_original.head())

# 3. Seleccionar una muestra aleatoria de 5.000 distritos
df = df_original.sample(
    n=5000,
    random_state=SEMILLA
).copy()

# 4. Reiniciar los índices para trabajar ordenadamente
df.reset_index(drop=True, inplace=True)

# 5. Verificar la muestra seleccionada
print("\nMUESTRA PERSONAL")
print("Semilla utilizada:", SEMILLA)
print("Número de distritos:", df.shape[0])
print("Número de variables:", df.shape[1])

# 6. Revisar los tipos de datos y los valores faltantes
print("\nTIPOS DE DATOS")
print(df.dtypes)

print("\nVALORES FALTANTES")
print(df.isnull().sum())

# 7. Revisar estadísticas descriptivas
print("\nESTADÍSTICAS DESCRIPTIVAS")
display(df.describe().round(2))


Las desviaciones estándar son especialmente altas en habitaciones, población y valor de vivienda, y sus máximos están muy por encima de los cuartiles superiores; conviene revisar esas colas y posibles valores atípicos en el siguiente paso.

## 1. Exploración y preparación

En esta parte vamos a describir las columnas, revisar observaciones extremas con el rango intercuartílico (IQR) y crear tres indicadores por hogar. Estos últimos ayudan a comparar distritos de tamaños distintos. Después estandarizaremos las variables elegidas para la clusterización, sin incluir `median_house_value`.

**Valores atípicos:** IQR = Q3 − Q1. Se señalarán observaciones menores que Q1 − 1.5×IQR o mayores que Q3 + 1.5×IQR. Es una señal para revisar la distribución, no una orden automática de borrar datos: los distritos muy grandes o costosos pueden ser reales.

In [ ]:
# Diccionario de variables y revisión inicial de valores atípicos
descripciones = {
    'longitude': 'Coordenada geográfica oeste-este del distrito (grados).',
    'latitude': 'Coordenada geográfica sur-norte del distrito (grados).',
    'housing_median_age': 'Edad mediana de las viviendas del distrito (años).',
    'total_rooms': 'Número total de habitaciones reportadas en el distrito.',
    'total_bedrooms': 'Número total de dormitorios reportados en el distrito.',
    'population': 'Población total del distrito.',
    'households': 'Número de hogares del distrito.',
    'median_income': 'Ingreso mediano del hogar; la escala del archivo está expresada en decenas de miles de dólares.',
    'median_house_value': 'Valor mediano de las viviendas del distrito (dólares); será variable objetivo en la regresión y no se usará para formar los clústeres.'
}
diccionario = pd.DataFrame({'Variable': descripciones.keys(), 'Descripción': descripciones.values()})
display(diccionario)

# Coordenadas se revisan geográficamente; el criterio IQR se aplica a las demás variables.
variables_iqr = [c for c in df.select_dtypes(include='number').columns if c not in ['longitude', 'latitude']]
filas = []
for columna in variables_iqr:
    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    mascara = (df[columna] < limite_inferior) | (df[columna] > limite_superior)
    filas.append({
        'Variable': columna, 'Q1': q1, 'Q3': q3, 'IQR': iqr,
        'Límite inferior': limite_inferior, 'Límite superior': limite_superior,
        'Posibles atípicos': int(mascara.sum()),
        'Porcentaje (%)': 100 * mascara.mean()
    })
resumen_iqr = pd.DataFrame(filas).set_index('Variable')
display(resumen_iqr.round(2))

# Diagramas de caja: muestran asimetría y posibles observaciones extremas.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(variables_iqr), 1, figsize=(10, 2.2 * len(variables_iqr)))
for ax, columna in zip(axes, variables_iqr):
    ax.boxplot(df[columna], vert=False, showfliers=True)
    ax.set_title(columna)
    ax.set_xlabel('Valor')
plt.tight_layout()
plt.show()

In [ ]:
# Crear indicadores relativos al tamaño del distrito
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['population_per_household'] = df['population'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']

variables_nuevas = ['rooms_per_household', 'population_per_household', 'bedrooms_per_room']
print('Variables derivadas (originales)')
display(df[variables_nuevas].describe().round(2))

# Winsorización: se limitan las razones a los percentiles 1 y 99 para que
# unos pocos distritos extremos no formen un grupo propio en K-means.
# Se conservan los 5.000 distritos; solo se recortan los valores extremos.
for col in variables_nuevas:
    p01, p99 = df[col].quantile([0.01, 0.99])
    df[col + '_w'] = df[col].clip(lower=p01, upper=p99)

variables_nuevas_w = [c + '_w' for c in variables_nuevas]
print('Variables derivadas (winsorizadas)')
display(df[variables_nuevas_w].describe().round(2))

### Variables derivadas y preparación para K-means

Creamos **habitaciones por hogar** (rooms_per_household), **personas por hogar** (population_per_household) y **dormitorios por habitación** (bedrooms_per_room). Las razones por hogar describen densidad y composición de los distritos sin depender solo de su tamaño total.

Para K-means usaré ubicación (longitude, latitude), edad de viviendas, ingreso mediano, habitaciones por hogar, personas por hogar y dormitorios por habitación. Excluyo los conteos absolutos porque reflejan mucho el tamaño del distrito, y excluyo median_house_value para que el precio no determine los grupos; luego lo podremos comparar entre grupos.

**Estandarización:** K-means agrupa según distancias. Si usamos las escalas originales, ingresos, coordenadas y conteos podrían pesar de forma muy distinta. StandardScaler transforma cada variable para que tenga media cercana a 0 y desviación estándar cercana a 1. Los valores transformados son puntuaciones z: indican cuántas desviaciones estándar está un distrito por encima o debajo de la media.

Las razones por hogar tienen valores extremos (hasta 141,9 habitaciones por hogar), y K-means es sensible a ellos. Por eso creé versiones limitadas a los percentiles 1 y 99 (terminan en _w) y usaré esas tanto en K-means como en la regresión. Así conservo los 5.000 distritos.

In [ ]:
from sklearn.preprocessing import StandardScaler

# median_house_value se reserva para describir los grupos después; no entra en K-means.
variables_cluster = [
    'longitude', 'latitude', 'housing_median_age', 'median_income',
    'rooms_per_household_w', 'population_per_household_w', 'bedrooms_per_room_w'
]
X_cluster = df[variables_cluster].copy()

# Estandarización: cada variable queda con media 0 y desviación estándar 1,
# para que ninguna domine las distancias de K-means por su escala.
escalador = StandardScaler()
X_estandarizado = pd.DataFrame(
    escalador.fit_transform(X_cluster),
    columns=variables_cluster,
    index=df.index
)

print('Variables usadas en K-means:', variables_cluster)
print('Dimensiones de X_estandarizado:', X_estandarizado.shape)
print('Medias después de estandarizar (cercanas a 0):')
display(X_estandarizado.mean().round(3).to_frame('media'))
print('Desviaciones estándar después de estandarizar (cercanas a 1):')
display(X_estandarizado.std(ddof=0).round(3).to_frame('desviación estándar'))

In [ ]:
# Revisar posibles valores atípicos de las variables derivadas
filas_nuevas = []
for columna in variables_nuevas:
    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)
    iqr = q3 - q1
    inferior = q1 - 1.5 * iqr
    superior = q3 + 1.5 * iqr
    mascara = (df[columna] < inferior) | (df[columna] > superior)
    filas_nuevas.append({
        'Variable': columna, 'Límite inferior': inferior,
        'Límite superior': superior, 'Posibles atípicos': int(mascara.sum()),
        'Porcentaje (%)': 100 * mascara.mean()
    })
resumen_iqr_nuevas = pd.DataFrame(filas_nuevas).set_index('Variable')
display(resumen_iqr_nuevas.round(2))

# Valores en el tope del archivo (datos censurados, no valores "reales")
print('Distritos con median_house_value en el tope ($500.001):', int((df['median_house_value'] >= 500001).sum()))
print('Distritos con median_income en el tope (15):', int((df['median_income'] >= 15).sum()))
print('Distritos con housing_median_age en el tope (52 años):', int((df['housing_median_age'] >= 52).sum()))

Con la regla del IQR, entre 3,7 % y 6,6 % de los distritos aparecen como atípicos en cada variable (la edad de las viviendas no tiene atípicos por este criterio). Casi todos son distritos muy grandes o muy caros, no errores. Distinto es el caso de las razones por hogar: 141,9 habitaciones o 41,2 personas por hogar no describen un hogar típico; probablemente son zonas con muchas casas vacacionales o residencias colectivas. Como K-means es sensible a los extremos, limité esas razones a los percentiles 1 y 99 y así conservé los 5.000 distritos. Además, hay datos topados en el archivo: 244 distritos tienen median_house_value en $500.001, 14 tienen median_income en 15 y 310 (6,2 %) tienen housing_median_age en 52 años. Es decir, parte de los valores “altos” son datos censurados.

##2. Clusterización

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Para cada k se calcula la inercia (codo) y la silueta con TODOS los distritos.
ks = range(2, 11)
resultados_k = []
for k in ks:
    modelo = KMeans(n_clusters=k, random_state=SEMILLA, n_init=10)
    etiquetas = modelo.fit_predict(X_estandarizado)
    silueta = silhouette_score(X_estandarizado, etiquetas)
    resultados_k.append({'k': k, 'Inercia (codo)': modelo.inertia_, 'Silueta': silueta})

resultados_k = pd.DataFrame(resultados_k)
display(resultados_k.round(3))

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
ejes[0].plot(resultados_k['k'], resultados_k['Inercia (codo)'], marker='o')
ejes[0].set(title='Método del codo', xlabel='Número de grupos (k)', ylabel='Inercia')
ejes[0].set_xticks(list(ks))
ejes[1].plot(resultados_k['k'], resultados_k['Silueta'], marker='o', color='darkorange')
ejes[1].set(title='Coeficiente de silueta', xlabel='Número de grupos (k)', ylabel='Silueta promedio')
ejes[1].set_xticks(list(ks))
plt.tight_layout()
plt.show()

Usé los 5.000 distritos, la semilla 5759 y siete variables estandarizadas; median_house_value quedó fuera de los grupos y solo sirvió para describirlos. En el método del codo, la mayor caída de inercia ocurre al pasar de k=2 a k=3 (de 26.109 a 20.801); después las caídas son pequeñas y graduales (2.451 de k=3 a k=4), así que el quiebre está en k=3. La silueta de k=3 (0,261) es casi igual a la de k=2 (0,269) y mayor que la de k=4 (0,238). k=2 solo separa norte y sur con ingreso casi igual (3,91 y 3,89), así que no sirve para decidir dónde invertir. Elegí k=3 porque es donde coinciden ambos criterios y porque entrega tres perfiles económicos distintos y fáciles de describir.

In [ ]:
k_final = 3  # k elegido: quiebre del codo en k=3 y silueta de 0,261

# Ajuste final de K-means con el k elegido
modelo_final = KMeans(n_clusters=k_final, random_state=SEMILLA, n_init=10)
df['cluster'] = modelo_final.fit_predict(X_estandarizado)

# Reetiquetar los grupos por ingreso medio: 0 = menor ingreso, ..., k-1 = mayor ingreso.
# Así el grupo 0 (la referencia de las dummies) siempre es el de menor ingreso.
orden = df.groupby('cluster')['median_income'].mean().sort_values().index
df['cluster'] = df['cluster'].map({viejo: nuevo for nuevo, viejo in enumerate(orden)})
silueta_final = resultados_k.loc[resultados_k['k'] == k_final, 'Silueta'].iloc[0]

# Perfil promedio de cada grupo (median_house_value solo se usa para describir)
perfil = df.groupby('cluster').agg(
    distritos=('cluster', 'size'),
    longitud=('longitude', 'mean'),
    latitud=('latitude', 'mean'),
    edad_vivienda=('housing_median_age', 'mean'),
    ingreso_mediano=('median_income', 'mean'),
    habitaciones_por_hogar=('rooms_per_household_w', 'mean'),
    personas_por_hogar=('population_per_household_w', 'mean'),
    dormitorios_por_habitacion=('bedrooms_per_room_w', 'mean'),
    valor_mediano_vivienda=('median_house_value', 'mean')
).round(2)
print(f'k = {k_final} | inercia = {modelo_final.inertia_:.2f} | silueta = {silueta_final:.3f}')
print(perfil.to_string())
print('Proporción de la muestra por grupo (%):')
print((100 * df['cluster'].value_counts(normalize=True).sort_index()).round(1).to_string())

In [ ]:
# Nombres de los grupos según el perfil promedio (ordenados de menor a mayor ingreso)
nombres_cluster = {
    0: 'Sur de California, ingreso medio-bajo y hogares más densos',
    1: 'Norte y centro de California, ingreso medio',
    2: 'Ingreso alto y viviendas amplias'
}

df['nombre_cluster'] = df['cluster'].map(nombres_cluster)
print('Promedios por segmento:')
print(df.groupby('nombre_cluster')[['median_income', 'rooms_per_household_w', 'population_per_household_w',
                                    'bedrooms_per_room_w', 'median_house_value']].mean().round(2).to_string())

# Mapa por grupo: cada panel resalta un grupo sobre el fondo gris de todos los distritos
fig, axes = plt.subplots(1, k_final, figsize=(6 * k_final, 6), sharex=True, sharey=True)
for ax, c in zip(axes, sorted(df['cluster'].unique())):
    ax.scatter(df['longitude'], df['latitude'], s=4, color='lightgray')
    m = df['cluster'] == c
    ax.scatter(df.loc[m, 'longitude'], df.loc[m, 'latitude'], s=10, alpha=0.7)
    ax.set_title(f'Grupo {c}: {nombres_cluster[c]}', fontsize=9)
    ax.set_xlabel('Longitud')
    ax.grid(alpha=0.2)
axes[0].set_ylabel('Latitud')
plt.suptitle(f'California: distritos agrupados con K-means (k={k_final})')
plt.tight_layout()
plt.show()

Los perfiles obtenidos son (ingreso en decenas de miles de dólares de 1990):

Resumen de Perfiles por Grupo

---

Grupo 0 — 1.947 distritos (38,9 %)
* **Perfil:** Sur de California, ingreso medio-bajo y hogares más densos.
* **Detalles:** Ingreso de `2,94` (unos $29.400), `3,12` personas por hogar, `4,46` habitaciones por hogar y la mayor proporción de dormitorios por habitación (`0,25`), lo que refleja viviendas más pequeñas.
* **Valor medio de vivienda:** **$179.506**

---

Grupo 1 — 1.800 distritos (36,0 %)
* **Perfil:** Norte y centro de California, ingreso medio.
* **Detalles:** Ingreso de `3,49`, hogares más holgados (`5,34` habitaciones y `2,72` personas por hogar).
* **Valor medio de vivienda:** **$186.068**

---

Grupo 2 — 1.253 distritos (25,1 %)
* **Perfil:** Ingreso alto y viviendas amplias.
* **Detalles:** Ingreso de `5,99` (unos $59.900), `6,72` habitaciones por hogar, la menor proporción de dormitorios por habitación (`0,16`) y las viviendas más nuevas (`23,8` años).
* **Valor medio de vivienda:** **$283.541** *(la más cara)*

---

Comparativa y Conclusiones del Agrupamiento

Los grupos se ordenan de manera escalonada: **a mayor ingreso, las viviendas son más amplias y más caras**.

Comparativa. Los grupos se ordenan de manera escalonada: a mayor ingreso, viviendas más amplias y más caras. El salto principal ocurre entre los grupos 1 y 2. Los grupos 0 y 1 tienen valores de vivienda muy similares ($179.506 y $186.068) y se distinguen más por la ubicación (sur frente a norte y centro) y la densidad del hogar que por el precio. El grupo 2 aparece tanto en la Bahía de San Francisco como en la costa de Los Ángeles, Orange y San Diego: el ingreso alto no depende de una sola región. Limitar los valores extremos de las razones por hogar evitó que unos pocos distritos formaran un grupo propio.

La silueta de k=3 es 0,261, apenas por encima de 0,25: indica una estructura débil, con grupos que se traslapan. Los distritos de California forman más un continuo que bloques claramente separados, pero tres perfiles resumen bien ese continuo.

##3. Regresión Multivariante

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm

# Variables explicativas (las razones por hogar son las versiones winsorizadas)
variables_base = ['median_income', 'housing_median_age', 'rooms_per_household_w',
                  'population_per_household_w', 'bedrooms_per_room_w']

# Modelo sin dummies de grupo
X_base = sm.add_constant(df[variables_base].astype(float))
modelo_base = sm.OLS(df['median_house_value'], X_base).fit()

# Modelo con dummies de grupo (categoría de referencia: grupo 0)
dummies_grupo = pd.get_dummies(df['cluster'].astype('category'), prefix='grupo', drop_first=True, dtype=float)
X_grupos = sm.add_constant(pd.concat([df[variables_base].astype(float), dummies_grupo], axis=1))
modelo_grupos = sm.OLS(df['median_house_value'], X_grupos).fit()

def tabla_coeficientes(modelo):
    tabla = pd.DataFrame({'coeficiente': modelo.params, 'error_std': modelo.bse, 't': modelo.tvalues,
                          'p_valor': modelo.pvalues, 'IC95_inf': modelo.conf_int()[0], 'IC95_sup': modelo.conf_int()[1]})
    tabla['significativo_5pct'] = tabla['p_valor'] < 0.05
    return tabla.round(4)

print('MODELO BASE: sin dummies de grupo')
print('R²:', round(modelo_base.rsquared, 4), '| R² ajustado:', round(modelo_base.rsquared_adj, 4))
print(tabla_coeficientes(modelo_base).to_string())
print('\nMODELO CON DUMMIES DE GRUPO (referencia: grupo 0)')
print('R²:', round(modelo_grupos.rsquared, 4), '| R² ajustado:', round(modelo_grupos.rsquared_adj, 4))
print(tabla_coeficientes(modelo_grupos).to_string())
print('\nComparación anidada (F clásico):')
print(anova_lm(modelo_base, modelo_grupos).to_string())

# Versión robusta a heterocedasticidad (HC3): mismos coeficientes, errores estándar corregidos
modelo_grupos_rob = sm.OLS(df['median_house_value'], X_grupos).fit(cov_type='HC3')
print('\nMODELO CON DUMMIES, ERRORES ROBUSTOS (HC3)')
print(tabla_coeficientes(modelo_grupos_rob).to_string())

# Prueba conjunta robusta: ¿las dummies de grupo son todas cero?
hipotesis = ', '.join(f'{c} = 0' for c in dummies_grupo.columns)
print('\nPrueba conjunta robusta de las dummies:')
print(modelo_grupos_rob.wald_test(hipotesis, use_f=True))

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera

# Multicolinealidad: VIF de los predictores del modelo con grupos
X_vif = pd.concat([df[variables_base].astype(float), dummies_grupo], axis=1)
vif = pd.DataFrame({'variable': X_vif.columns,
                    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]})
print('FACTORES DE INFLACIÓN DE VARIANZA (VIF)')
print(vif.round(2).to_string(index=False))

# Pruebas sobre los residuos
residuos = modelo_grupos.resid
ajustados = modelo_grupos.fittedvalues
bp = het_breuschpagan(residuos, modelo_grupos.model.exog)
jb = jarque_bera(residuos)
print('\nDIAGNÓSTICOS DE RESIDUOS')
print(f'Breusch-Pagan: LM={bp[0]:.2f}, p={bp[1]:.4g}; p<0.05 sugiere heterocedasticidad.')
print(f'Jarque-Bera: estadístico={jb[0]:.2f}, p={jb[1]:.4g}; p<0.05 sugiere residuos no normales.')

# Distritos con mayor influencia (distancia de Cook) y valores ajustados negativos
cooks = modelo_grupos.get_influence().cooks_distance[0]
print('\nLos 5 distritos más influyentes (distancia de Cook):')
print(pd.Series(cooks, index=df.index).nlargest(5).round(4).to_string())
print('Distritos con valor ajustado negativo:', int((ajustados < 0).sum()))

# Gráficos: residuos vs ajustados y Q-Q (fit=True estandariza los residuos)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(ajustados, residuos, s=10, alpha=0.35)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set(title='Residuos frente a valores ajustados', xlabel='Valor ajustado', ylabel='Residuo')
sm.qqplot(residuos, line='45', fit=True, ax=axes[1], alpha=0.4)
axes[1].set_title('Gráfico Q-Q de residuos (estandarizados)')
plt.tight_layout()
plt.show()

## Interpretación de la regresión y diagnósticos

Se estimaron dos modelos OLS para predecir el valor mediano de vivienda: uno sin dummies de grupo y otro con dummies de los tres grupos de K-means. La referencia es el Grupo 0 (Sur de California, ingreso medio-bajo y hogares más densos). Los coeficientes expresan asociaciones manteniendo fijas las demás variables, no efectos causales. En este conjunto, median_income está expresado en decenas de miles de dólares y median_house_value en dólares del periodo del censo.

### Coeficientes del modelo con dummies

- **Ingreso mediano:** +USD 50.856 por una unidad de median_income (p < 0,001; también significativo con HC3). Una unidad equivale aproximadamente a USD 10.000 en la escala de ingreso del archivo.
- **Edad mediana de vivienda:** +USD 1.392 por año (p < 0,001; también con HC3). Es una asociación condicional; no significa que el envejecimiento cause que una vivienda se encarezca.
- **Habitaciones por hogar winsorizada:** −USD 4.702 por habitación adicional (p = 0,0014; p = 0,0155 con HC3). El signo es contraintuitivo y debe interpretarse junto con los demás indicadores de composición del hogar.
- **Personas por hogar winsorizada:** −USD 39.213 por persona adicional (p < 0,001; también con HC3). Es compatible con una asociación entre mayor densidad y menor valor, sin implicar causalidad.
- **Dormitorios por habitación winsorizada:** +USD 469.503 por unidad de proporción (p < 0,001; también con HC3). La unidad completa es grande: un aumento de 0,01 se asocia con aproximadamente +USD 4.695, manteniendo lo demás fijo. El signo condicional puede diferir de la comparación simple entre grupos.
- **Constante:** +USD 13.870 (p = 0,383); no es significativa y no tiene una interpretación práctica útil porque corresponde a predictores iguales a cero.

### Dummies de grupo y ajuste

Con el Grupo 0 como referencia, la dummy **Grupo 1** tiene coeficiente −USD 12.346 (p < 0,001; HC3 p < 0,001): con los predictores controlados, el valor esperado es menor que el del Grupo 0. La dummy **Grupo 2** es +USD 2.166 (p = 0,578; HC3 p = 0,604), por lo que no hay evidencia de una diferencia propia frente al Grupo 0 una vez controladas las otras variables.

El modelo base tiene R² ajustado **0,6018** y el modelo con dummies **0,6042**: la mejora es **0,0024** (0,24 puntos porcentuales). La prueba F clásica de las dummies da F = 16,27, p < 0,001; la prueba conjunta robusta HC3 da F = 15,16, p < 0,001. En conjunto las dummies aportan información estadísticamente detectable, aunque la ganancia explicativa es pequeña. No debe confundirse significancia estadística con una gran mejora predictiva.

### Diagnósticos y supuestos

- **Multicolinealidad:** la celda de diagnóstico calcula el VIF para cada predictor del modelo con grupos. Interprétese la tabla que aparece en esa salida: VIF cercanos a 1 indican poca asociación lineal y valores superiores a 5 ameritan revisar redundancia. Las razones por hogar comparten componentes y las dummies se derivan de variables incluidas en el análisis, por lo que cierta asociación es esperable.
- **Varianza constante:** Breusch–Pagan contrasta homocedasticidad. Un p < 0,05 permite rechazarla; junto con un patrón de abanico en residuos frente a ajustados, indica heterocedasticidad. Por eso se presentan también errores estándar robustos HC3.
- **Normalidad:** Jarque–Bera y el gráfico Q–Q evalúan si los residuos siguen una distribución normal. Un p < 0,05 o desviaciones claras de la diagonal en las colas indican falta de normalidad.
- **Linealidad y forma funcional:** revise el gráfico de residuos y el conteo de valores ajustados negativos impreso por el código. Un patrón sistemático o predicciones negativas mostrarían limitaciones del modelo lineal en dólares; podría probarse el logaritmo del valor de vivienda.
- **Independencia:** no queda garantizada por OLS; al ser distritos geográficos, puede existir dependencia espacial.
- **Influencia:** la celda imprime los cinco valores más altos de distancia de Cook. Esos registros deben revisarse antes de afirmar que alguna observación domina el ajuste.

Como los diagnósticos de residuos pueden invalidar los errores estándar clásicos, las conclusiones de significancia se apoyan principalmente en los p-valores HC3 de la tabla.

##4. Lo que probé y descarté

In [ ]:
from sklearn.metrics import adjusted_rand_score

# --- Alternativa 1: k=2 con las mismas variables estandarizadas ---
modelo_k2 = KMeans(n_clusters=2, random_state=SEMILLA, n_init=10)
df['cluster_k2'] = modelo_k2.fit_predict(X_estandarizado)
silueta_k2 = silhouette_score(X_estandarizado, df['cluster_k2'])

perfil_k2 = df.groupby('cluster_k2').agg(
    distritos=('cluster_k2', 'size'),
    longitud=('longitude', 'mean'),
    latitud=('latitude', 'mean'),
    ingreso_mediano=('median_income', 'mean'),
    personas_por_hogar=('population_per_household_w', 'mean'),
    valor_vivienda=('median_house_value', 'mean')
).round(2)
print('Alternativa k=2:')
print(perfil_k2.to_string())
print(f'Inercia k=2: {modelo_k2.inertia_:.2f}; silueta k=2: {silueta_k2:.3f}')
print(f'Modelo elegido k={k_final}: inercia {modelo_final.inertia_:.2f}; silueta {silueta_final:.3f}')

fig, ax = plt.subplots(figsize=(9, 6))
for g in sorted(df['cluster_k2'].unique()):
    m = df['cluster_k2'] == g
    ax.scatter(df.loc[m, 'longitude'], df.loc[m, 'latitude'], s=13, alpha=0.6, label=f'Grupo k=2: {g}')
ax.set(title='Alternativa descartada: K-means con k=2', xlabel='Longitud', ylabel='Latitud')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# --- Alternativa 2: mismo k, pero SIN estandarizar ---
modelo_sin_std = KMeans(n_clusters=k_final, random_state=SEMILLA, n_init=10)
df['cluster_sin_std'] = modelo_sin_std.fit_predict(X_cluster)  # X_cluster = variables sin estandarizar

print('\nDesviación estándar de cada variable SIN estandarizar (la de mayor escala domina las distancias):')
print(X_cluster.std().round(2).to_string())
print('\nPerfil de los grupos sin estandarizar:')
print(df.groupby('cluster_sin_std').agg(
    distritos=('cluster_sin_std', 'size'),
    ingreso_mediano=('median_income', 'mean'),
    edad_vivienda=('housing_median_age', 'mean'),
    latitud=('latitude', 'mean'),
    valor_vivienda=('median_house_value', 'mean')).round(2).to_string())
print('\nConcordancia con la solución estandarizada (índice de Rand ajustado, 1 = idéntica):',
      round(adjusted_rand_score(df['cluster'], df['cluster_sin_std']), 3))

## Alternativas evaluadas y decisión

### Alternativa 1: K-means con k=2
Se mantuvieron las mismas variables estandarizadas y la semilla 5759; solo cambió el número de grupos.

| Grupo | Distritos | Ingreso mediano promedio | Valor mediano promedio (USD) |
|---|---:|---:|---:|
| 0 | 2.940 (58,8 %) | 3,91 | 212.012 |
| 1 | 2.060 (41,2 %) | 3,89 | 202.127 |

La silueta de k=2 fue 0,269, ligeramente mayor que la de k=3 (0,261), por lo que k=2 obtiene grupos algo más compactos y separados según esta métrica. Sin embargo, sus promedios de ingreso son casi iguales y ofrece una segmentación geográfica/económica demasiado general. Para orientar decisiones de inversión se conserva k=3: el codo muestra una reducción marcada de inercia hasta k=3 y los tres perfiles resultan más interpretables. La elección prioriza utilidad de negocio e interpretabilidad, aceptando una silueta apenas menor.

### Alternativa 2: K-means sin estandarizar
También se probó k=3 con las mismas variables en sus escalas originales. Se descartó porque las escalas difieren mucho (por ejemplo, edad de vivienda: desviación estándar 12,58; ingreso: 1,92; dormitorios por habitación winsorizado: 0,05). Así, las variables de mayor escala dominan las distancias. La partición resultó muy distinta de la estandarizada (índice de Rand ajustado = 0,02) y separó principalmente por edad de vivienda, con perfiles menos alineados con el objetivo de segmentar por ingreso, composición y ubicación.

**Decisión final:** usar k=3 con las variables estandarizadas. La solución sin estandarizar muestra por qué la escala debe controlarse; k=2 es una alternativa estadísticamente razonable por silueta, pero menos útil para distinguir perfiles accionables.

##5. Recomendación


**Conclusión aplicada (hipótesis, no recomendación de inversión)**

El Grupo 1 presenta un valor medio descriptivo cercano a 186 mil dólares y una diferencia condicional estimada de aproximadamente 12.300 dólares menos que el Grupo 0, controlando por las variables incluidas en la regresión. Esto puede motivar un análisis adicional, pero no demuestra infravaloración ni rentabilidad superior.

Para evaluar una inversión harían falta datos de alquileres, costos, vacancia, demanda y evolución temporal. Los datos corresponden a 1990; los resultados son asociaciones y no prueban causalidad. Los identificadores de grupo son etiquetas arbitrarias de K-means.

##7. Uso de IA

Se utilizó la IA, para la generación del código con las librerías que implementan el análisis descriptivo de las variables, winsorizar y estandarizar las variables, realizar la clusterización y la regresión. Fue una herramienta muy útil pero las interpretaciones, justificaciones y comprobaciones de los resultados se resolvieron con los conocimientos adqueridos en la clase de Estadistíca 4.